In [1]:
import torch
import torch.nn as nn
from transformers import AutoModel, AutoTokenizer, AutoConfig

class SimpleCostModel(nn.Module):
    def __init__(self, model_name="distilgpt2"):
        """
        Initializes a simple cost model using a pre-trained transformer.

        Args:
            model_name (str): Name of the pre-trained transformer model
                                from Hugging Face (e.g., "distilgpt2", "gpt2").
        """
        super().__init__()
        self.config = AutoConfig.from_pretrained(model_name)
        self.transformer = AutoModel.from_pretrained(model_name)
        # Add a linear layer to map the transformer's hidden size to a single score
        self.score_head = nn.Linear(self.config.hidden_size, 1)

    def forward(self, input_ids: torch.LongTensor, attention_mask: torch.BoolTensor) -> torch.Tensor:
        """
        Forward pass to compute the cost score.

        Args:
            input_ids (torch.LongTensor): Token IDs of the input sequence (prompt + response). Shape: (batch_size, sequence_length).
            attention_mask (torch.BoolTensor): Mask indicating non-padding tokens. Shape: (batch_size, sequence_length).

        Returns:
            torch.Tensor: The predicted cost score for each sequence. Shape: (batch_size, 1).
        """
        # Get hidden states from the base transformer model
        outputs = self.transformer(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=False, # We only need the last layer's hidden states
        )
        # Get the hidden states of the last layer
        last_hidden_state = outputs.last_hidden_state # Shape: (batch_size, sequence_length, hidden_size)

        # Find the index of the last non-padding token for each sequence
        # attention_mask is 1 for real tokens, 0 for padding. sum(dim=1) gives sequence lengths.
        sequence_lengths = attention_mask.sum(dim=1) - 1 # Subtract 1 for 0-based indexing
        # Gather the hidden state of the last token for each sequence
        # We use gather along dim=1 (sequence_length dimension)
        # Need to reshape sequence_lengths to match the dimensions for gather
        last_token_hidden_state = torch.gather(
            last_hidden_state,
            dim=1,
            index=sequence_lengths.unsqueeze(-1).unsqueeze(-1).expand(-1, -1, last_hidden_state.shape[-1])
        ).squeeze(1) # Shape: (batch_size, hidden_size)

        # Pass the last token's hidden state through the linear head
        scores = self.score_head(last_token_hidden_state) # Shape: (batch_size, 1)

        return scores

In [5]:
from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader
import random

# Choose the same tokenizer as your model
MODEL_NAME = "distilgpt2" # Or "gpt2", etc. Must match SimpleCostModel
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Set padding token if it doesn't exist (common for GPT-2)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

MAX_LENGTH = 512 # Adjust based on your model and memory

def preprocess_beaver_tails(examples):
    """Preprocesses a batch of examples from BeaverTails."""
    processed = {
        "safer_input_ids": [], "safer_attention_mask": [], "safer_safety_sign": [],
        "unsafer_input_ids": [], "unsafer_attention_mask": [], "unsafer_safety_sign": [],
    }

    prompts = examples["prompt"]
    responses_0 = examples["response_0"]
    responses_1 = examples["response_1"]
    is_response_0_safe = examples["is_response_0_safe"]

    for i in range(len(prompts)):
        prompt = prompts[i]
        res0 = responses_0[i]
        res1 = responses_1[i]

        # Combine prompt and response
        text0 = prompt + tokenizer.eos_token + res0 # Add EOS token as separator
        text1 = prompt + tokenizer.eos_token + res1

        # Tokenize
        tokenized0 = tokenizer(text0, max_length=MAX_LENGTH, padding="max_length", truncation=True, return_tensors=None) # Return lists for now
        tokenized1 = tokenizer(text1, max_length=MAX_LENGTH, padding="max_length", truncation=True, return_tensors=None)

        if is_response_0_safe[i]:
            # response 0 is safer, response 1 is unsafer
            processed["safer_input_ids"].append(tokenized0["input_ids"])
            processed["safer_attention_mask"].append(tokenized0["attention_mask"])
            processed["safer_safety_sign"].append(1) # +1 for safe

            processed["unsafer_input_ids"].append(tokenized1["input_ids"])
            processed["unsafer_attention_mask"].append(tokenized1["attention_mask"])
            processed["unsafer_safety_sign"].append(-1) # -1 for unsafe
        else:
            # response 1 is safer, response 0 is unsafer
            processed["safer_input_ids"].append(tokenized1["input_ids"])
            processed["safer_attention_mask"].append(tokenized1["attention_mask"])
            processed["safer_safety_sign"].append(1)

            processed["unsafer_input_ids"].append(tokenized0["input_ids"])
            processed["unsafer_attention_mask"].append(tokenized0["attention_mask"])
            processed["unsafer_safety_sign"].append(-1)

    return processed

# Load the dataset (adjust split name if needed, e.g., '330k_train', '330k_test')
# Using a smaller subset for demonstration might be faster initially
try:
    # Try loading specific splits often used
    train_dataset = load_dataset("PKU-Alignment/PKU-SafeRLHF", split="330k_train")
    eval_dataset = load_dataset("PKU-Alignment/PKU-SafeRLHF", split="330k_test")
except Exception as e:
    print(f"Could not load specific splits, trying 'train': {e}")
    # Fallback to default 'train' split, might need manual splitting later
    full_dataset = load_dataset("PKU-Alignment/PKU-SafeRLHF", split="train")
    # Simple 90/10 split - consider stratified split if class balance is important
    split_dataset = full_dataset.train_test_split(test_size=0.1, seed=42)
    train_dataset = split_dataset['train']
    eval_dataset = split_dataset['test']


# Apply preprocessing - use map for efficiency
# Consider using num_proc > 1 if you have multiple cores
train_processed = train_dataset.map(preprocess_beaver_tails, batched=True, remove_columns=train_dataset.column_names)
eval_processed = eval_dataset.map(preprocess_beaver_tails, batched=True, remove_columns=eval_dataset.column_names)

# Set format for PyTorch
train_processed.set_format(type='torch', columns=['safer_input_ids', 'safer_attention_mask', 'safer_safety_sign', 'unsafer_input_ids', 'unsafer_attention_mask', 'unsafer_safety_sign'])
eval_processed.set_format(type='torch', columns=['safer_input_ids', 'safer_attention_mask', 'safer_safety_sign', 'unsafer_input_ids', 'unsafer_attention_mask', 'unsafer_safety_sign'])

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

Generating train split:   0%|          | 0/73907 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/8211 [00:00<?, ? examples/s]

Could not load specific splits, trying 'train': Unknown split "330k_train". Should be one of ['train', 'test'].


Map:   0%|          | 0/66516 [00:00<?, ? examples/s]

Map:   0%|          | 0/7391 [00:00<?, ? examples/s]

In [4]:
pip install datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 30.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 16.1 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.2 requires fsspec==2025.3.2, but you have fsspec 2024.12.0 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which 

In [6]:
import torch.nn.functional as F

def compute_loss(safer_end_scores, unsafer_end_scores, safer_safety_sign, unsafer_safety_sign, regularization_coeff=0.01):
    """
    Computes the sequence-wise loss for the cost model.

    Args:
        safer_end_scores (torch.Tensor): Scores for the safer sequences (batch_size, 1).
        unsafer_end_scores (torch.Tensor): Scores for the unsafer sequences (batch_size, 1).
        safer_safety_sign (torch.Tensor): Safety signs for safer sequences (+1). (batch_size,)
        unsafer_safety_sign (torch.Tensor): Safety signs for unsafer sequences (-1). (batch_size,)
        regularization_coeff (float): Coefficient for L2 regularization on scores.

    Returns:
        torch.Tensor: The calculated scalar loss.
        torch.Tensor: Accuracy of ranking (unsafer > safer).
        torch.Tensor: Accuracy of predicted cost sign.
    """
    # Squeeze scores to shape (batch_size,)
    lower_end_cost = safer_end_scores.squeeze(-1)
    higher_end_cost = unsafer_end_scores.squeeze(-1)

    # Derive target cost sign (-1 for safe, +1 for unsafe)
    lower_cost_sign = -safer_safety_sign.float()
    higher_cost_sign = -unsafer_safety_sign.float()

    # Ranking loss: encourage higher_end_cost > lower_end_cost
    ranking_loss = -F.logsigmoid(higher_end_cost - lower_end_cost)

    # Sign loss: encourage costs to have the correct sign
    # Note: Use logsigmoid(sign * cost). If sign matches cost sign -> large positive -> logsigmoid -> ~0 loss
    # If sign mismatches cost sign -> large negative -> logsigmoid -> large negative -> large positive loss
    lower_sign_loss = -F.logsigmoid(lower_cost_sign * lower_end_cost)
    higher_sign_loss = -F.logsigmoid(higher_cost_sign * higher_end_cost)

    # Combine losses
    loss = (ranking_loss + lower_sign_loss + higher_sign_loss).mean()

    # Optional Regularization
    if regularization_coeff > 0.0:
        l2_reg = torch.stack([lower_end_cost, higher_end_cost]).square().mean()
        loss = loss + regularization_coeff * l2_reg

    # Calculate accuracies for monitoring
    accuracy_ranking = (higher_end_cost > lower_end_cost).float().mean()
    accuracy_sign = torch.cat(
        [
            (lower_cost_sign * lower_end_cost > 0.0), # Check if cost has correct sign
            (higher_cost_sign * higher_end_cost > 0.0)
        ],
        dim=0 # Concatenate along batch dimension before mean
    ).float().mean()


    return loss, accuracy_ranking, accuracy_sign

In [ ]:
from torch.optim import AdamW
from tqdm.auto import tqdm # For progress bars

# Hyperparameters (adjust as needed)
LEARNING_RATE = 1e-5
BATCH_SIZE = 8 # Reduce if you run out of memory
NUM_EPOCHS = 1 # Start with 1 epoch for testing
REGULARIZATION = 0.01

# Setup device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Instantiate model and move to device
model = SimpleCostModel(model_name=MODEL_NAME).to(device)

# Create DataLoaders
train_dataloader = DataLoader(train_processed, batch_size=BATCH_SIZE, shuffle=True)
# You would typically create an eval_dataloader as well for validation
# eval_dataloader = DataLoader(eval_processed, batch_size=BATCH_SIZE)

# Setup optimizer
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)

# --- Training Loop ---
model.train() # Set model to training mode
for epoch in range(NUM_EPOCHS):
    print(f"--- Epoch {epoch+1}/{NUM_EPOCHS} ---")
    total_loss = 0.0
    total_acc_rank = 0.0
    total_acc_sign = 0.0

    progress_bar = tqdm(train_dataloader, desc=f"Epoch {epoch+1}")

    for batch in progress_bar:
        # Move batch to device
        safer_ids = batch['safer_input_ids'].to(device)
        safer_mask = batch['safer_attention_mask'].to(device)
        safer_sign = batch['safer_safety_sign'].to(device)
        unsafer_ids = batch['unsafer_input_ids'].to(device)
        unsafer_mask = batch['unsafer_attention_mask'].to(device)
        unsafer_sign = batch['unsafer_safety_sign'].to(device)

        # Zero gradients
        optimizer.zero_grad()

        # Forward pass for both safer and unsafer inputs
        safer_scores = model(input_ids=safer_ids, attention_mask=safer_mask)
        unsafer_scores = model(input_ids=unsafer_ids, attention_mask=unsafer_mask)

        # Compute loss
        loss, acc_rank, acc_sign = compute_loss(
            safer_scores, unsafer_scores,
            safer_sign, unsafer_sign,
            regularization_coeff=REGULARIZATION
        )

        # Backward pass
        loss.backward()

        # Optimizer step
        optimizer.step()

        # Update tracking variables
        total_loss += loss.item()
        total_acc_rank += acc_rank.item()
        total_acc_sign += acc_sign.item()

        # Update progress bar description (optional)
        progress_bar.set_postfix({
            'loss': loss.item(),
            'acc_rank': acc_rank.item(),
            'acc_sign': acc_sign.item()
        })

    # Calculate average metrics for the epoch
    avg_loss = total_loss / len(train_dataloader)
    avg_acc_rank = total_acc_rank / len(train_dataloader)
    avg_acc_sign = total_acc_sign / len(train_dataloader)
    print(f"Epoch {epoch+1} finished. Avg Loss: {avg_loss:.4f}, Avg Acc Rank: {avg_acc_rank:.4f}, Avg Acc Sign: {avg_acc_sign:.4f}")

    # --- Add evaluation loop here ---
    # model.eval()
    # with torch.no_grad():
    #     for eval_batch in eval_dataloader:
    #         # ... compute eval metrics ...
    # model.train() # Set back to train mode

# --- Save the model (optional) ---
# torch.save(model.state_dict(), f"{MODEL_NAME}_cost_model.pth")
# print("Model saved.")

Using device: cuda


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

--- Epoch 1/1 ---


Epoch 1:   0%|          | 0/8315 [00:00<?, ?it/s]

In [10]:
pip install hf_xet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 MB 12.1 MB/s eta 0:00:00
